# Project 8: Food Delivery Operations Analytics
**Industry:** FoodTech / Logistics  
**Dataset:** `08_food_delivery.csv`  

### Business Scenario
Evaluate delivery times, order outcomes, restaurant performance, and city-level operations across 500 food delivery orders to identify service bottlenecks and formulate actionable operational improvements.

---

In [1]:
# Environment & Library Setup
import pandas as pd
import numpy as np

# Optional visualization libraries (notebook runs all tasks even if not installed)
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_theme(style='whitegrid', palette='muted')
    plt.rcParams['figure.figsize'] = (10, 5)
    plt.rcParams['figure.dpi'] = 110
    HAS_PLOTS = True
except ImportError:
    HAS_PLOTS = False
    print("Note: matplotlib/seaborn not installed. Running in pure pandas/numpy mode (run '%pip install matplotlib seaborn' if you want charts).")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Core libraries (pandas, numpy) loaded successfully.")

Core libraries (pandas, numpy) loaded successfully.


### Task 109: Load and inspect the order dataset

In [3]:
# Task 109: Load and inspect the order dataset
df = pd.read_csv('08_food_delivery.csv')

print(f"Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns\n")
print("--- Data Types & Non-Null Counts ---")
df.info()

print("\n--- First 5 Rows ---")
print(df.head())

print("\n--- Summary Statistics (All Columns) ---")
print(df.describe(include='all'))

Dataset Shape: 500 rows x 9 columns

--- Data Types & Non-Null Counts ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Order_ID           500 non-null    object 
 1   City               500 non-null    object 
 2   Restaurant         500 non-null    object 
 3   Order_Value        500 non-null    float64
 4   Distance_km        500 non-null    float64
 5   Delivery_Time_Min  493 non-null    float64
 6   Order_Status       500 non-null    object 
 7   Payment_Mode       500 non-null    object 
 8   Order_Hour         500 non-null    int64  
dtypes: float64(3), int64(1), object(5)
memory usage: 35.3+ KB

--- First 5 Rows ---
  Order_ID       City   Restaurant  Order_Value  Distance_km  Delivery_Time_Min Order_Status Payment_Mode  Order_Hour
0  FD00000      Noida   Green Bowl       588.00         2.60              32.00    Delivere

### Task 110: Find missing delivery times and decide how to handle them

In [5]:
# Task 110: Find missing delivery times and decide how to handle them
print("--- Missing Values Across All Columns ---")
missing_summary = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Pct (%)': (df.isnull().sum() / len(df) * 100).round(2)
})
print(missing_summary)

missing_dt_rows = df[df['Delivery_Time_Min'].isnull()]
print(f"\nRecords with Missing Delivery_Time_Min ({len(missing_dt_rows)} rows):")
print(missing_dt_rows)

# Decision & Imputation:
# All 7 missing records are valid 'Delivered' orders (1.40% of dataset).
# Dropping them would lose valid revenue and order counts. Because delivery time has
# right-skewed outliers, median imputation grouped by City preserves city-specific
# delivery baselines without distortion from extreme delays.
city_medians = df.groupby('City')['Delivery_Time_Min'].median()
print("\nCity-wise Median Delivery Time (before imputation):")
print(city_medians)

df['Delivery_Time_Min'] = df['Delivery_Time_Min'].fillna(
    df.groupby('City')['Delivery_Time_Min'].transform('median')
)

print(f"\nMissing Delivery_Time_Min after City-wise median imputation: {df['Delivery_Time_Min'].isnull().sum()}")

--- Missing Values Across All Columns ---
                   Missing_Count  Missing_Pct (%)
Order_ID                       0             0.00
City                           0             0.00
Restaurant                     0             0.00
Order_Value                    0             0.00
Distance_km                    0             0.00
Delivery_Time_Min              7             1.40
Order_Status                   0             0.00
Payment_Mode                   0             0.00
Order_Hour                     0             0.00

Records with Missing Delivery_Time_Min (7 rows):
    Order_ID       City   Restaurant  Order_Value  Distance_km  Delivery_Time_Min Order_Status Payment_Mode  Order_Hour
38   FD00038  Faridabad   Green Bowl       196.38         4.40                NaN    Delivered         Card          10
54   FD00054      Delhi      Cafe 24       335.92         6.90                NaN    Delivered         Card          13
58   FD00058      Noida   Green Bowl       309.9

### Task 111: Check duplicate `Order_ID` values

In [7]:
# Task 111: Check duplicate Order_ID values
dup_order_ids = df['Order_ID'].duplicated().sum()
dup_full_rows = df.duplicated().sum()
unique_orders = df['Order_ID'].nunique()

print(f"Total Records          : {len(df)}")
print(f"Unique Order_IDs       : {unique_orders}")
print(f"Duplicate Order_IDs    : {dup_order_ids}")
print(f"Duplicate Full Rows    : {dup_full_rows}")

if dup_order_ids > 0:
    print("\nDuplicate Order_ID rows found:")
    print(df[df['Order_ID'].duplicated(keep=False)])
    df = df.drop_duplicates(subset=['Order_ID'], keep='first').reset_index(drop=True)
    print(f"Duplicates removed. New row count: {len(df)}")
else:
    print("\nVerification Passed: All 500 Order_ID values are unique.")

Total Records          : 500
Unique Order_IDs       : 500
Duplicate Order_IDs    : 0
Duplicate Full Rows    : 0

Verification Passed: All 500 Order_ID values are unique.


### Task 112: Calculate total orders and order-status percentages

In [9]:
# Task 112: Calculate total orders and order-status percentages
total_orders = len(df)
status_counts = df['Order_Status'].value_counts()
status_pct = df['Order_Status'].value_counts(normalize=True) * 100

status_summary = pd.DataFrame({
    'Order_Count': status_counts,
    'Percentage (%)': status_pct.round(2)
})

print(f"Total Orders: {total_orders}\n")
print(status_summary)

if HAS_PLOTS:
    fig, ax = plt.subplots(figsize=(7, 4))
    colors = ['#2a9d8f', '#e9c46a', '#e76f51']
    bars = ax.bar(status_summary.index, status_summary['Order_Count'], color=colors, width=0.5)
    for bar, pct in zip(bars, status_summary['Percentage (%)']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 6,
                f"{int(bar.get_height())} ({pct:.1f}%)", ha='center', va='bottom', fontweight='bold')
    ax.set_title('Order Volume & Share by Order Status (n = 500)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Orders')
    ax.set_ylim(0, 470)
    plt.tight_layout()
    plt.show()

Total Orders: 500

              Order_Count  Percentage (%)
Order_Status                             
Delivered             407           81.40
Delayed                47            9.40
Cancelled              46            9.20


### Task 113: Calculate average delivery time by city

In [11]:
# Task 113: Calculate average delivery time by city
city_delivery = (
    df.groupby('City')['Delivery_Time_Min']
    .agg(
        Order_Count='count',
        Avg_Delivery_Min='mean',
        Median_Delivery_Min='median',
        Std_Dev_Min='std',
        Min_Min='min',
        Max_Min='max'
    )
    .round(2)
    .sort_values(by='Avg_Delivery_Min', ascending=False)
)

print("--- Delivery Time Summary by City ---")
print(city_delivery)

if HAS_PLOTS:
    fig, ax = plt.subplots(figsize=(8.5, 4.4))
    overall_avg = df['Delivery_Time_Min'].mean()
    bars = ax.barh(city_delivery.index[::-1], city_delivery['Avg_Delivery_Min'][::-1], color='#457b9d', height=0.5)
    ax.axvline(overall_avg, color='#e63946', linestyle='--', linewidth=1.5, label=f'Network Avg: {overall_avg:.2f} min')
    for bar in bars:
        ax.text(bar.get_width() + 0.6, bar.get_y() + bar.get_height()/2,
                f"{bar.get_width():.2f} min", va='center', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='none', alpha=0.85))
    ax.set_title('Average Delivery Time by City', fontsize=12, fontweight='bold', pad=24)
    ax.set_xlabel('Average Delivery Time (Minutes)')
    ax.set_xlim(0, 45)
    ax.legend(bbox_to_anchor=(0.5, 1.01), loc='lower center', frameon=True)
    plt.tight_layout()
    plt.show()

--- Delivery Time Summary by City ---
           Order_Count  Avg_Delivery_Min  Median_Delivery_Min  Std_Dev_Min  Min_Min  Max_Min
City                                                                                        
Noida              125             37.45                36.00        11.72    14.00    75.00
Gurugram           115             36.09                36.00        11.95    10.00    67.00
Delhi              119             36.04                36.00        10.80    10.00    64.00
Faridabad          141             35.18                35.00        11.98    10.00    71.00


### Task 114: Find the five restaurants with highest total order value

In [13]:
# Task 114: Find the five restaurants with highest total order value
top5_restaurants = (
    df.groupby('Restaurant')
    .agg(
        Total_Orders=('Order_ID', 'count'),
        Total_Order_Value=('Order_Value', 'sum'),
        Revenue_Share_Pct=('Order_Value', lambda x: x.sum() / df['Order_Value'].sum() * 100)
    )
    .round(2)
    .sort_values(by='Total_Order_Value', ascending=False)
    .head(5)
)

print("--- Top 5 Restaurants by Total Order Value ---")
print(top5_restaurants)

--- Top 5 Restaurants by Total Order Value ---
             Total_Orders  Total_Order_Value  Revenue_Share_Pct
Restaurant                                                     
Pizza Point           114           50662.50              22.25
Green Bowl            107           49773.47              21.86
Urban Bites           100           46168.30              20.27
Spice Hub              92           41464.49              18.21
Cafe 24                87           39652.52              17.41


### Task 115: Calculate average order value by restaurant

In [15]:
# Task 115: Calculate average order value by restaurant
aov_by_restaurant = (
    df.groupby('Restaurant')
    .agg(
        Total_Orders=('Order_ID', 'count'),
        Total_Revenue=('Order_Value', 'sum'),
        Avg_Order_Value=('Order_Value', 'mean'),
        Median_Order_Value=('Order_Value', 'median'),
        Std_Order_Value=('Order_Value', 'std')
    )
    .round(2)
    .sort_values(by='Avg_Order_Value', ascending=False)
)

print("--- Average Order Value (AOV) by Restaurant ---")
print(aov_by_restaurant)

if HAS_PLOTS:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    rev_sorted = aov_by_restaurant.sort_values(by='Total_Revenue', ascending=True)
    axes[0].barh(rev_sorted.index, rev_sorted['Total_Revenue'], color='#2a9d8f', height=0.5)
    for i, v in enumerate(rev_sorted['Total_Revenue']):
        axes[0].text(v + 600, i, f"₹{v:,.0f}", va='center', fontsize=9, fontweight='bold')
    axes[0].set_title('Total Revenue by Restaurant', fontweight='bold')
    axes[0].set_xlabel('Total Order Value (₹)')
    axes[0].set_xlim(0, 59000)

    aov_sorted = aov_by_restaurant.sort_values(by='Avg_Order_Value', ascending=True)
    axes[1].barh(aov_sorted.index, aov_sorted['Avg_Order_Value'], color='#f4a261', height=0.5)
    for i, v in enumerate(aov_sorted['Avg_Order_Value']):
        axes[1].text(v + 5, i, f"₹{v:.2f}", va='center', fontsize=9, fontweight='bold')
    axes[1].set_title('Average Order Value (AOV) by Restaurant', fontweight='bold')
    axes[1].set_xlabel('Average Order Value (₹)')
    axes[1].set_xlim(0, 530)

    plt.tight_layout()
    plt.show()

--- Average Order Value (AOV) by Restaurant ---
             Total_Orders  Total_Revenue  Avg_Order_Value  Median_Order_Value  Std_Order_Value
Restaurant                                                                                    
Green Bowl            107       49773.47           465.17              486.05           165.88
Urban Bites           100       46168.30           461.68              447.12           140.46
Cafe 24                87       39652.52           455.78              463.14           179.00
Spice Hub              92       41464.49           450.70              452.30           173.12
Pizza Point           114       50662.50           444.41              438.01           185.69


### Task 116: Identify peak ordering hours

In [17]:
# Task 116: Identify peak ordering hours
hourly_summary = (
    df.groupby('Order_Hour')
    .agg(
        Order_Count=('Order_ID', 'count'),
        Order_Share_Pct=('Order_ID', lambda x: len(x) / len(df) * 100),
        Total_Revenue=('Order_Value', 'sum'),
        Avg_Delivery_Min=('Delivery_Time_Min', 'mean')
    )
    .round(2)
)

print("--- Top 5 Peak Ordering Hours (Sorted by Order Count) ---")
print(hourly_summary.sort_values(by='Order_Count', ascending=False).head(5))

print("\n--- Full Hourly Operations Breakdown (08:00 to 23:00) ---")
print(hourly_summary)

if HAS_PLOTS:
    fig, ax1 = plt.subplots(figsize=(11, 4.5))
    hours = hourly_summary.index
    counts = hourly_summary['Order_Count']
    bar_colors = ['#e63946' if c >= 36 else '#457b9d' for c in counts]

    bars = ax1.bar(hours, counts, color=bar_colors, alpha=0.85, label='Order Count (Red = Peak >=36)')
    ax1.set_xlabel('Order Hour of Day (24-Hour Clock)')
    ax1.set_ylabel('Order Count', color='#1d3557')
    ax1.set_xticks(hours)
    ax1.set_ylim(0, 45)

    for bar in bars:
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.6,
                 f"{int(bar.get_height())}", ha='center', va='bottom', fontsize=8)

    ax2 = ax1.twinx()
    ax2.plot(hours, hourly_summary['Avg_Delivery_Min'], color='#f4a261', marker='o', linewidth=2, label='Avg Delivery Time (min)')
    ax2.set_ylabel('Avg Delivery Time (Minutes)', color='#d67d29')
    ax2.set_ylim(25, 48)
    ax2.grid(False)

    plt.title('Hourly Order Volume & Average Delivery Time', fontsize=12, fontweight='bold')
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
    plt.tight_layout()
    plt.show()

--- Top 5 Peak Ordering Hours (Sorted by Order Count) ---
            Order_Count  Order_Share_Pct  Total_Revenue  Avg_Delivery_Min
Order_Hour                                                               
10                   38             7.60       17558.53             37.16
18                   38             7.60       17911.39             36.79
21                   37             7.40       16106.52             35.30
13                   36             7.20       18247.69             35.78
8                    34             6.80       14383.97             35.38

--- Full Hourly Operations Breakdown (08:00 to 23:00) ---
            Order_Count  Order_Share_Pct  Total_Revenue  Avg_Delivery_Min
Order_Hour                                                               
8                    34             6.80       14383.97             35.38
9                    34             6.80       14625.02             35.91
10                   38             7.60       17558.53             3

### Task 117: Compare delivery time across order-status categories

In [19]:
# Task 117: Compare delivery time across order-status categories
status_delivery_comparison = (
    df.groupby('Order_Status')['Delivery_Time_Min']
    .agg(
        Order_Count='count',
        Mean_Min='mean',
        Median_Min='median',
        Std_Dev_Min='std',
        Min_Min='min',
        Q25_Min=lambda x: np.percentile(x, 25),
        Q75_Min=lambda x: np.percentile(x, 75),
        Max_Min='max'
    )
    .round(2)
)

print("--- Delivery Time Comparison Across Order-Status Categories ---")
print(status_delivery_comparison)

if HAS_PLOTS:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.boxplot(
        data=df, x='Order_Status', y='Delivery_Time_Min',
        order=['Delivered', 'Delayed', 'Cancelled'],
        palette=['#2a9d8f', '#e9c46a', '#e76f51'],
        width=0.45, ax=ax
    )
    ax.set_title('Delivery Time Distribution by Order Status', fontsize=12, fontweight='bold')
    ax.set_xlabel('Order Status')
    ax.set_ylabel('Delivery Time (Minutes)')
    plt.tight_layout()
    plt.show()

--- Delivery Time Comparison Across Order-Status Categories ---
              Order_Count  Mean_Min  Median_Min  Std_Dev_Min  Min_Min  Q25_Min  Q75_Min  Max_Min
Order_Status                                                                                    
Cancelled              46     36.52       38.50        11.43    10.00    29.00    45.75    53.00
Delayed                47     36.19       35.00        12.75    13.00    26.50    46.00    62.00
Delivered             407     36.12       35.00        11.55    10.00    29.00    43.00    75.00


### Task 118: Calculate cancellation and delay rates by city

In [21]:
# Task 118: Calculate cancellation and delay rates by city
city_status_counts = pd.crosstab(df['City'], df['Order_Status'])
city_status_pct = pd.crosstab(df['City'], df['Order_Status'], normalize='index') * 100

city_rates = pd.DataFrame({
    'Total_Orders': city_status_counts.sum(axis=1),
    'Delivered_Count': city_status_counts['Delivered'],
    'Delayed_Count': city_status_counts['Delayed'],
    'Cancelled_Count': city_status_counts['Cancelled'],
    'Delivered_Rate (%)': city_status_pct['Delivered'].round(2),
    'Delay_Rate (%)': city_status_pct['Delayed'].round(2),
    'Cancellation_Rate (%)': city_status_pct['Cancelled'].round(2),
    'Combined_Issue_Rate (%)': (city_status_pct['Delayed'] + city_status_pct['Cancelled']).round(2)
}).sort_values(by='Combined_Issue_Rate (%)', ascending=False)

print("--- Cancellation and Delay Rates by City ---")
print(city_rates)

if HAS_PLOTS:
    ax = city_rates[['Delivered_Rate (%)', 'Delay_Rate (%)', 'Cancellation_Rate (%)']].plot(
        kind='bar', stacked=True, figsize=(9, 4.5),
        color=['#2a9d8f', '#e9c46a', '#e76f51'], width=0.55
    )
    plt.title('Order Outcome Breakdown (%) by City', fontsize=12, fontweight='bold')
    plt.ylabel('Percentage of City Orders (%)')
    plt.xlabel('City')
    plt.xticks(rotation=0)
    plt.legend(title='Order Status', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

--- Cancellation and Delay Rates by City ---
           Total_Orders  Delivered_Count  Delayed_Count  Cancelled_Count  Delivered_Rate (%)  Delay_Rate (%)  Cancellation_Rate (%)  Combined_Issue_Rate (%)
City                                                                                                                                                        
Gurugram            115               91             11               13               79.13            9.57                  11.30                    20.87
Delhi               119               95             12               12               79.83           10.08                  10.08                    20.17
Noida               125              100             13               12               80.00           10.40                   9.60                    20.00
Faridabad           141              121             11                9               85.82            7.80                   6.38                    14.18


### Task 119: Use NumPy to calculate median and standard deviation of delivery time

In [23]:
# Task 119: Use NumPy to calculate median and standard deviation of delivery time
delivery_times_np = df['Delivery_Time_Min'].to_numpy()

np_median = np.median(delivery_times_np)
np_mean = np.mean(delivery_times_np)
np_std_pop = np.std(delivery_times_np, ddof=0)     # Population standard deviation (NumPy default)
np_std_sample = np.std(delivery_times_np, ddof=1)  # Sample standard deviation

print("--- NumPy Statistical Summary of Delivery Time ---")
print(f"Mean Delivery Time                   : {np_mean:.2f} minutes")
print(f"Median Delivery Time (np.median)     : {np_median:.2f} minutes")
print(f"Std Dev - Population (np.std, ddof=0): {np_std_pop:.2f} minutes")
print(f"Std Dev - Sample (np.std, ddof=1)    : {np_std_sample:.2f} minutes")

--- NumPy Statistical Summary of Delivery Time ---
Mean Delivery Time                   : 36.16 minutes
Median Delivery Time (np.median)     : 36.00 minutes
Std Dev - Population (np.std, ddof=0): 11.62 minutes
Std Dev - Sample (np.std, ddof=1)    : 11.64 minutes


### Task 120: Identify unusually long deliveries using IQR

In [25]:
# Task 120: Identify unusually long deliveries using IQR
dt_arr = df['Delivery_Time_Min'].to_numpy()

q1 = np.percentile(dt_arr, 25)
q3 = np.percentile(dt_arr, 75)
iqr = q3 - q1

lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

unusually_long_df = df[df['Delivery_Time_Min'] > upper_fence].sort_values(
    by='Delivery_Time_Min', ascending=False
)

print("--- IQR Outlier Detection Summary ---")
print(f"25th Percentile (Q1)        : {q1:.2f} min")
print(f"75th Percentile (Q3)        : {q3:.2f} min")
print(f"Interquartile Range (IQR)   : {iqr:.2f} min")
print(f"Lower Outlier Fence         : {lower_fence:.2f} min")
print(f"Upper Outlier Fence         : {upper_fence:.2f} min")
print(f"Unusually Long Deliveries   : {len(unusually_long_df)} orders (> {upper_fence:.2f} min)\n")
print(unusually_long_df[['Order_ID', 'City', 'Restaurant', 'Order_Value', 'Distance_km', 'Delivery_Time_Min', 'Order_Status', 'Order_Hour']])

if HAS_PLOTS:
    fig, ax = plt.subplots(figsize=(9, 3.2))
    sns.boxplot(x=df['Delivery_Time_Min'], color='#8ecae6', ax=ax)
    ax.axvline(upper_fence, color='#e63946', linestyle='--', linewidth=1.8, label=f'Upper IQR Fence ({upper_fence:.1f} min)')
    ax.set_title('Delivery Time Boxplot & Upper IQR Outlier Threshold', fontsize=11, fontweight='bold')
    ax.set_xlabel('Delivery Time (Minutes)')
    ax.legend()
    plt.tight_layout()
    plt.show()

--- IQR Outlier Detection Summary ---
25th Percentile (Q1)        : 29.00 min
75th Percentile (Q3)        : 44.00 min
Interquartile Range (IQR)   : 15.00 min
Lower Outlier Fence         : 6.50 min
Upper Outlier Fence         : 66.50 min
Unusually Long Deliveries   : 3 orders (> 66.50 min)

    Order_ID       City   Restaurant  Order_Value  Distance_km  Delivery_Time_Min Order_Status  Order_Hour
454  FD00454      Noida    Spice Hub       497.76         6.90              75.00    Delivered          20
57   FD00057  Faridabad    Spice Hub       479.34         3.50              71.00    Delivered          10
174  FD00174   Gurugram  Pizza Point       366.60         2.10              67.00    Delivered          14


### Task 121: Create delivery-time bands such as Fast, Standard and Delayed

In [27]:
# Task 121: Create delivery-time bands such as Fast, Standard and Delayed
conditions = [
    df['Delivery_Time_Min'] <= 30,
    (df['Delivery_Time_Min'] > 30) & (df['Delivery_Time_Min'] <= 45),
    df['Delivery_Time_Min'] > 45
]
band_labels = ['Fast (<=30 min)', 'Standard (31-45 min)', 'Delayed (>45 min)']

df['Delivery_Band'] = np.select(conditions, band_labels, default='Unknown')

band_summary = pd.DataFrame({
    'Order_Count': df['Delivery_Band'].value_counts(),
    'Percentage (%)': (df['Delivery_Band'].value_counts(normalize=True) * 100).round(2),
    'Avg_Order_Value': df.groupby('Delivery_Band')['Order_Value'].mean().round(2),
    'Avg_Distance_km': df.groupby('Delivery_Band')['Distance_km'].mean().round(2)
}).reindex(band_labels)

print("--- Delivery-Time Band Distribution ---")
print(band_summary)

print("\n--- Delivery-Time Bands by City (%) ---")
band_by_city = (pd.crosstab(df['City'], df['Delivery_Band'], normalize='index') * 100).round(2)[band_labels]
print(band_by_city)

--- Delivery-Time Band Distribution ---
                      Order_Count  Percentage (%)  Avg_Order_Value  Avg_Distance_km
Delivery_Band                                                                      
Fast (<=30 min)               151           30.20           458.09             5.13
Standard (31-45 min)          246           49.20           466.45             5.27
Delayed (>45 min)             103           20.60           425.28             4.87

--- Delivery-Time Bands by City (%) ---
Delivery_Band  Fast (<=30 min)  Standard (31-45 min)  Delayed (>45 min)
City                                                                   
Delhi                    26.89                 57.98              15.13
Faridabad                34.04                 45.39              20.57
Gurugram                 28.70                 50.43              20.87
Noida                    30.40                 44.00              25.60


### Task 122: Group revenue by city and payment mode

In [29]:
# Task 122: Group revenue by city and payment mode
rev_grouped = (
    df.groupby(['City', 'Payment_Mode'])
    .agg(
        Order_Count=('Order_ID', 'count'),
        Total_Revenue=('Order_Value', 'sum'),
        Avg_Order_Value=('Order_Value', 'mean')
    )
    .round(2)
)
print("--- Revenue Grouped by City and Payment Mode ---")
print(rev_grouped)

rev_pivot = df.pivot_table(
    values='Order_Value',
    index='City',
    columns='Payment_Mode',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
).round(2)
print("\n--- Pivot Table: Total Revenue (₹) by City & Payment Mode ---")
print(rev_pivot)

if HAS_PLOTS:
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    sns.heatmap(
        rev_pivot.iloc[:-1, :-1], annot=True, fmt=',.2f', cmap='YlGnBu',
        cbar_kws={'label': 'Total Revenue (₹)'}, ax=ax
    )
    ax.set_title('Total Revenue (₹) by City and Payment Mode', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

--- Revenue Grouped by City and Payment Mode ---
                        Order_Count  Total_Revenue  Avg_Order_Value
City      Payment_Mode                                             
Delhi     Card                   38       18242.29           480.06
          Cash                   38       17373.79           457.21
          UPI                    43       19298.39           448.80
Faridabad Card                   62       27959.61           450.96
          Cash                   47       20585.91           438.00
          UPI                    32       12414.03           387.94
Gurugram  Card                   35       16440.50           469.73
          Cash                   43       21349.48           496.50
          UPI                    37       17616.62           476.12
Noida     Card                   44       20550.72           467.06
          Cash                   38       16832.78           442.97
          UPI                    43       19057.16           443.19

In [1]:
# Task 123: Operational Recommendations
print("""=== TASK 123: OPERATIONAL RECOMMENDATIONS ===

1. Deploy Peak-Hour & Early-Morning Rider Surge Scheduling (08:00, 10:00, 13:00, 18:00-21:00)
   - Finding: Order demand peaks at 10:00 AM (38 orders), 6:00 PM (38 orders), 9:00 PM (37 orders),
     and 1:00 PM (36 orders). Meanwhile, 8:00 PM has the slowest average delivery time (41.36 min),
     9:00 PM hits an 18.92% cancellation rate, and 8:00 AM suffers a 20.59% delay rate and 14.71% cancellation rate.
   - Action: Schedule staggered rider micro-shifts and surge incentives 30 minutes prior to the
     08:00 AM morning rush and 18:00-21:00 dinner peak to prevent rider shortages and late-night cancellations.

2. Target High-Issue Cities (Gurugram, Noida, and Delhi) with Localized Dispatch & COD Controls
   - Finding: Faridabad leads with an 85.82% on-time delivery rate (35.18 min avg delivery, 6.38% cancellation),
     whereas Gurugram has the highest cancellation rate (11.30%; 20.87% combined delay/cancel rate) and high Cash
     volume (Rs. 21,349.48). Noida records the slowest average delivery time (37.45 min) and highest delay rate (10.40%).
   - Action: Tighten dispatch zone radii in Noida and Delhi (10.08% delay, 10.08% cancel) to cut transit delays,
     and introduce OTP/pre-confirmation nudges on high-value Cash orders in Gurugram to reduce cancellations.

3. Resolve Restaurant-Specific Bottlenecks at Pizza Point and Green Bowl
   - Finding: Pizza Point generates the highest revenue (Rs. 50,662.50 across 114 orders) but has a 14.91%
     cancellation rate (nearly 1 in 7 orders cancelled) and the lowest AOV (Rs. 444.41). Green Bowl generates
     the second-highest revenue (Rs. 49,773.47) and highest AOV (Rs. 465.17) but has the slowest average
     delivery time (37.71 min) and highest delay rate (12.15%).
   - Action: Audit kitchen-prep SLAs with Green Bowl (triggering rider dispatch closer to food-ready time) and
     Pizza Point (managing peak-hour capacity/stockouts), while adding combo upsells at Pizza Point to lift AOV.

4. Automate SLA-Based Status Tagging & Escalate Long Deliveries (>45 Min / Outliers >66.5 Min)
   - Finding: 'Delivered' orders average 36.12 min (up to 75.0 min), nearly identical to 'Delayed' orders
     (36.19 min). Overall, 20.60% of orders (103 orders) exceed 45 minutes, and IQR analysis (upper fence = 66.50 min)
     identifies 3 extreme outliers: FD00454 (75.0 min), FD00057 (71.0 min), and FD00174 (67.0 min).
   - Action: Automatically flag any active order crossing the 45-minute SLA as 'Delayed' for proactive customer
     alerts, and trigger live ops escalation when any delivery exceeds 50 minutes.

5. Enforce Mandatory Delivery Timestamp Logging & Incentivize UPI Adoption in Faridabad
   - Finding: 1.40% of orders (7 'Delivered' records, 4 in Delhi) were missing Delivery_Time_Min values.
     Additionally, Faridabad leads total revenue (Rs. 60,959.55) via Card and Cash but lags in UPI adoption
     (32 orders, lowest AOV at Rs. 387.94).
   - Action: Require a mandatory completion timestamp in the rider app before closing an order, and run targeted
     UPI cashback offers in Faridabad to shift Cash orders toward faster digital payments.
""")

=== TASK 123: OPERATIONAL RECOMMENDATIONS ===

1. Deploy Peak-Hour & Early-Morning Rider Surge Scheduling (08:00, 10:00, 13:00, 18:00-21:00)
   - Finding: Order demand peaks at 10:00 AM (38 orders), 6:00 PM (38 orders), 9:00 PM (37 orders),
     and 1:00 PM (36 orders). Meanwhile, 8:00 PM has the slowest average delivery time (41.36 min),
     9:00 PM hits an 18.92% cancellation rate, and 8:00 AM suffers a 20.59% delay rate and 14.71% cancellation rate.
   - Action: Schedule staggered rider micro-shifts and surge incentives 30 minutes prior to the
     08:00 AM morning rush and 18:00-21:00 dinner peak to prevent rider shortages and late-night cancellations.

2. Target High-Issue Cities (Gurugram, Noida, and Delhi) with Localized Dispatch & COD Controls
   - Finding: Faridabad leads with an 85.82% on-time delivery rate (35.18 min avg delivery, 6.38% cancellation),
     whereas Gurugram has the highest cancellation rate (11.30%; 20.87% combined delay/cancel rate) and high Cash
     volum

In [31]:
# Consolidated Executive Deliverable Tables (City Performance & Restaurant Ranking)
city_exec = (
    df.groupby('City')
    .agg(
        Total_Orders=('Order_ID', 'count'),
        Total_Revenue=('Order_Value', 'sum'),
        Avg_Order_Value=('Order_Value', 'mean'),
        Avg_Delivery_Min=('Delivery_Time_Min', 'mean'),
        Delay_Rate_Pct=('Order_Status', lambda s: (s == 'Delayed').mean() * 100),
        Cancel_Rate_Pct=('Order_Status', lambda s: (s == 'Cancelled').mean() * 100)
    )
    .round(2)
    .sort_values(by='Total_Revenue', ascending=False)
)

rest_exec = (
    df.groupby('Restaurant')
    .agg(
        Total_Orders=('Order_ID', 'count'),
        Total_Revenue=('Order_Value', 'sum'),
        Avg_Order_Value=('Order_Value', 'mean'),
        Avg_Delivery_Min=('Delivery_Time_Min', 'mean'),
        Delay_Rate_Pct=('Order_Status', lambda s: (s == 'Delayed').mean() * 100),
        Cancel_Rate_Pct=('Order_Status', lambda s: (s == 'Cancelled').mean() * 100)
    )
    .round(2)
    .sort_values(by='Total_Revenue', ascending=False)
)

print("=== CONSOLIDATED CITY PERFORMANCE TABLE ===")
print(city_exec)

print("\n=== CONSOLIDATED RESTAURANT PERFORMANCE RANKING ===")
print(rest_exec)

=== CONSOLIDATED CITY PERFORMANCE TABLE ===
           Total_Orders  Total_Revenue  Avg_Order_Value  Avg_Delivery_Min  Delay_Rate_Pct  Cancel_Rate_Pct
City                                                                                                      
Faridabad           141       60959.55           432.34             35.18            7.80             6.38
Noida               125       56440.66           451.53             37.45           10.40             9.60
Gurugram            115       55406.60           481.80             36.09            9.57            11.30
Delhi               119       54914.47           461.47             36.04           10.08            10.08

=== CONSOLIDATED RESTAURANT PERFORMANCE RANKING ===
             Total_Orders  Total_Revenue  Avg_Order_Value  Avg_Delivery_Min  Delay_Rate_Pct  Cancel_Rate_Pct
Restaurant                                                                                                  
Pizza Point           114       50662.50   